In [1]:
# In[1] — thread caps (fork-safety / reproducibility)
import os
for v in ('OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS',
          'VECLIB_MAXIMUM_THREADS','NUMEXPR_NUM_THREADS'):
    os.environ[v] = '1'

In [2]:
# In[2] — imports, path, reload, confirm the passthrough is in your repo
import sys, os, importlib, warnings, inspect
warnings.simplefilter(action='ignore', category=FutureWarning)
sys.path.insert(0, os.path.abspath('../../model'))
sys.path.insert(0, os.path.abspath('.'))
sys.path.insert(0, os.path.abspath('../.'))

In [3]:
import os
for v in ('OMP','OPENBLAS','MKL','VECLIB','NUMEXPR'): os.environ[f'{v}_NUM_THREADS']='1'
import importlib, numpy as np, seasonal_scan_harness as ssh; importlib.reload(ssh)
FORC = ssh.build_forcings(['pre+recovery','post'], hplc_coincident=False)
RF_PRE = [0.3, 0.4, 0.5, 0.55, 0.6, 0.65]
KSZ=[0.15,0.20,0.23]; GGE=[0.25,0.28,0.31,0.33]; SIG=[0.15,0.18,0.20,0.22,0.25]; MZ=[0.05,0.10]
DETR=[(5.0,0.04),(5.0,0.067),(7.5,0.04)]                       # (w_sink,k_remin) paired -> L 125/75/188
PIN=dict(mP=0.0015, fish_fX=0.25, fish_fD=0.25, zq_fD=0.90, zq_fX=0.0, zl_fD=0.90, zl_fX=0.0)
COMBOS=[dict(KsZ=k,GGE=g,sigma_log=s,m_Z=m,w_sink=ws,k_remin=kr,**PIN)
        for k in KSZ for g in GGE for s in SIG for m in MZ for (ws,kr) in DETR]   # 360 constructs
print(f"{len(COMBOS)} constructs x {len(RF_PRE)} r_F = {len(COMBOS)*len(RF_PRE)} pre runs")
res = ssh.run_seasonal_scan(constructs=['maranon_ward'], groups=['pre+recovery'], fish_rates=RF_PRE,
    param_combos=COMBOS, years=60, spinup=15, n_harmonics=3, forcings=FORC, routed=True,
    save_path='seasonal_grid2_pre_2026-06-29.pkl', processes=None, maxtasksperchild=1)
print('pre done:', len(res.records))

360 constructs x 6 r_F = 2160 pre runs
[seasonal scan] 2160 runs: ['maranon_ward'] x ['pre+recovery'] x fish=[0.3, 0.4, 0.5, 0.55, 0.6, 0.65] x params['GGE', 'KsZ', 'fish_fD', 'fish_fX', 'k_remin', 'mP', 'm_Z', 'sigma_log', 'w_sink', 'zl_fD', 'zl_fX', 'zq_fD', 'zq_fX'] | 60 yr each (spin-up 15)
solver: {'method': 'RK45', 'atol': 1e-09, 'rtol': 1e-06, 'max_step': 1.0, 'instability_neg_threshold': -0.001} | fourier n_harm=3 | processes=cpu-1
  MODEL   : model_baseline_seasonal_routed   (ROUTED closure+fish)
  growth  : maranon_ward  mu_max[0.13-0.83]  Ks[0.04-10.45]
  grazing : KsZ 0.15  sigma_log 0.15  Q10 2.48
  scalars : GGE 0.25  growth-Q10 1.62  T_ref 20.0
  rates   : mP 0.0015  m_Z(quad) 0.05  m_Zlin 0.05  r_F 0.3
  detritus: k_remin 0.04  w_sink 5.0  ->  L = 125 m
  loss-fate routing (N / D / export):
     phyto mort : N 0.10 / D 0.90 / export 0.00
     graze unas : N 0.25 / D 0.75 / export 0.00
     zoo linear : N 0.10 / D 0.90 / export 0.00
     zoo quad   : N 0.10 / D 0.90 / ex

/Users/aoop/Documents/GitHub/xarray-simlab-ode/src/xso/core.py:148: UserWarning: Instability event triggered at t=1587 on Z[1] = -0.00146. Run terminated, remaining time points NaN-padded. To loosen the safety threshold pass solver_kwargs={'instability_neg_threshold': -1e-3} to xso.setup; to disable, pass float('-inf').
  self.solver.solve(self.model, time_step)


   53/2160 maranon_ward  pre+recovery  fish=0.3  {'KsZ': 0.15, 'GGE': 0.31, 'sigma_log': 0.18, 'm_Z': 0.05, 'w_sink': 5.0, 'k_remin': 0.04, 'mP': 0.0015, 'fish_fX': 0.25, 'fish_fD': 0.25, 'zq_fD': 0.9, 'zq_fX': 0.0, 'zl_fD': 0.9, 'zl_fX': 0.0} mcs=  nan/  nan micro= nan Z200pk=  nan conv= nan nan=True score=  nan |  4.8m ETA 191.2m


/Users/aoop/Documents/GitHub/SSMCariaco/parameter_scan/seasonal_scan_harness.py:374: RuntimeWarning: Mean of empty slice
  clim['NPP'] = float(np.nanmean(r['NPP'][_k])); clim['export_total'] = float(np.nanmean(r['ExportTotal'][_k]))


   54/2160 maranon_ward  pre+recovery  fish=0.3  {'KsZ': 0.15, 'GGE': 0.28, 'sigma_log': 0.22, 'm_Z': 0.05, 'w_sink': 5.0, 'k_remin': 0.04, 'mP': 0.0015, 'fish_fX': 0.25, 'fish_fD': 0.25, 'zq_fD': 0.9, 'zq_fX': 0.0, 'zl_fD': 0.9, 'zl_fX': 0.0} mcs= 6.03/ 2.70 micro=0.26 Z200pk=0.313 conv=1.02 nan=False score= 0.40 |  4.8m ETA 188.9m
   55/2160 maranon_ward  pre+recovery  fish=0.3  {'KsZ': 0.15, 'GGE': 0.28, 'sigma_log': 0.22, 'm_Z': 0.05, 'w_sink': 7.5, 'k_remin': 0.04, 'mP': 0.0015, 'fish_fX': 0.25, 'fish_fD': 0.25, 'zq_fD': 0.9, 'zq_fX': 0.0, 'zl_fD': 0.9, 'zl_fX': 0.0} mcs= 5.19/ 2.62 micro=0.25 Z200pk=0.151 conv=1.20 nan=False score= 0.40 |  5.0m ETA 189.5m
   56/2160 maranon_ward  pre+recovery  fish=0.3  {'KsZ': 0.15, 'GGE': 0.28, 'sigma_log': 0.22, 'm_Z': 0.1, 'w_sink': 7.5, 'k_remin': 0.04, 'mP': 0.0015, 'fish_fX': 0.25, 'fish_fD': 0.25, 'zq_fD': 0.9, 'zq_fX': 0.0, 'zl_fD': 0.9, 'zl_fX': 0.0} mcs= 5.53/ 3.10 micro=0.27 Z200pk=0.238 conv=0.98 nan=False score= 0.32 |  5.0m ETA 188

...

  399/2160 maranon_ward  pre+recovery  fish=0.4  {'KsZ': 0.15, 'GGE': 0.28, 'sigma_log': 0.18, 'm_Z': 0.1, 'w_sink': 5.0, 'k_remin': 0.04, 'mP': 0.0015, 'fish_fX': 0.25, 'fish_fD': 0.25, 'zq_fD': 0.9, 'zq_fX': 0.0, 'zl_fD': 0.9, 'zl_fX': 0.0} mcs=  nan/  nan micro= nan Z200pk=  nan conv= nan nan=True score=  nan | 30.9m ETA 136.3m
  400/2160 maranon_ward  pre+recovery  fish=0.4  {'KsZ': 0.15, 'GGE': 0.28, 'sigma_log': 0.18, 'm_Z': 0.05, 'w_sink': 7.5, 'k_remin': 0.04, 'mP': 0.0015, 'fish_fX': 0.25, 'fish_fD': 0.25, 'zq_fD': 0.9, 'zq_fX': 0.0, 'zl_fD': 0.9, 'zl_fX': 0.0} mcs= 7.74/ 3.17 micro=0.31 Z200pk=0.200 conv=1.03 nan=False score= 0.31 | 30.9m ETA 136.1m
  401/2160 maranon_ward  pre+recovery  fish=0.4  {'KsZ': 0.15, 'GGE': 0.28, 'sigma_log': 0.18, 'm_Z': 0.1, 'w_sink': 7.5, 'k_remin': 0.04, 'mP': 0.0015, 'fish_fX': 0.25, 'fish_fD': 0.25, 'zq_fD': 0.9, 'zq_fX': 0.0, 'zl_fD': 0.9, 'zl_fX': 0.0} mcs= 8.77/ 3.20 micro=0.33 Z200pk=0.138 conv=1.03 nan=False score= 0.30 | 30.9m ETA 135.8

/Users/aoop/Documents/GitHub/xarray-simlab-ode/src/xso/core.py:148: UserWarning: Instability event triggered at t=1948 on Z[1] = -0.00175. Run terminated, remaining time points NaN-padded. To loosen the safety threshold pass solver_kwargs={'instability_neg_threshold': -1e-3} to xso.setup; to disable, pass float('-inf').
  self.solver.solve(self.model, time_step)


  1522/2160 maranon_ward  pre+recovery  fish=0.6  {'KsZ': 0.15, 'GGE': 0.33, 'sigma_log': 0.18, 'm_Z': 0.05, 'w_sink': 5.0, 'k_remin': 0.067, 'mP': 0.0015, 'fish_fX': 0.25, 'fish_fD': 0.25, 'zq_fD': 0.9, 'zq_fX': 0.0, 'zl_fD': 0.9, 'zl_fX': 0.0} mcs=  nan/  nan micro= nan Z200pk=  nan conv= nan nan=True score=  nan | 116.0m ETA 48.6m


/Users/aoop/Documents/GitHub/SSMCariaco/parameter_scan/seasonal_scan_harness.py:374: RuntimeWarning: Mean of empty slice
  clim['NPP'] = float(np.nanmean(r['NPP'][_k])); clim['export_total'] = float(np.nanmean(r['ExportTotal'][_k]))


  1523/2160 maranon_ward  pre+recovery  fish=0.6  {'KsZ': 0.15, 'GGE': 0.31, 'sigma_log': 0.25, 'm_Z': 0.05, 'w_sink': 5.0, 'k_remin': 0.04, 'mP': 0.0015, 'fish_fX': 0.25, 'fish_fD': 0.25, 'zq_fD': 0.9, 'zq_fX': 0.0, 'zl_fD': 0.9, 'zl_fX': 0.0} mcs= 5.18/ 4.27 micro=0.34 Z200pk=0.091 conv=1.03 nan=False score= 0.25 | 116.0m ETA 48.5m
  1524/2160 maranon_ward  pre+recovery  fish=0.6  {'KsZ': 0.15, 'GGE': 0.31, 'sigma_log': 0.22, 'm_Z': 0.1, 'w_sink': 7.5, 'k_remin': 0.04, 'mP': 0.0015, 'fish_fX': 0.25, 'fish_fD': 0.25, 'zq_fD': 0.9, 'zq_fX': 0.0, 'zl_fD': 0.9, 'zl_fX': 0.0} mcs= 8.04/ 3.74 micro=0.33 Z200pk=0.181 conv=0.99 nan=False score= 0.22 | 116.1m ETA 48.5m
  1525/2160 maranon_ward  pre+recovery  fish=0.6  {'KsZ': 0.15, 'GGE': 0.31, 'sigma_log': 0.22, 'm_Z': 0.1, 'w_sink': 5.0, 'k_remin': 0.04, 'mP': 0.0015, 'fish_fX': 0.25, 'fish_fD': 0.25, 'zq_fD': 0.9, 'zq_fX': 0.0, 'zl_fD': 0.9, 'zl_fX': 0.0} mcs= 6.66/ 3.88 micro=0.33 Z200pk=0.143 conv=1.07 nan=False score= 0.20 | 116.1m ETA 

In [4]:
import os
for v in ('OMP','OPENBLAS','MKL','VECLIB','NUMEXPR'): os.environ[f'{v}_NUM_THREADS']='1'
import importlib, numpy as np, seasonal_scan_harness as ssh; importlib.reload(ssh)
FORC = ssh.build_forcings(['pre+recovery','post'], hplc_coincident=False)
RF_POST = [0.1, 0.15, 0.2]
KSZ=[0.15,0.20,0.23]; GGE=[0.25,0.28,0.31,0.33]; SIG=[0.15,0.18,0.20,0.22,0.25]; MZ=[0.05,0.10]
DETR=[(5.0,0.04),(5.0,0.067),(7.5,0.04)]
PIN=dict(mP=0.0015, fish_fX=0.25, fish_fD=0.25, zq_fD=0.90, zq_fX=0.0, zl_fD=0.90, zl_fX=0.0)
COMBOS=[dict(KsZ=k,GGE=g,sigma_log=s,m_Z=m,w_sink=ws,k_remin=kr,**PIN)
        for k in KSZ for g in GGE for s in SIG for m in MZ for (ws,kr) in DETR]
print(f"{len(COMBOS)} constructs x {len(RF_POST)} r_F = {len(COMBOS)*len(RF_POST)} post runs")
res = ssh.run_seasonal_scan(constructs=['maranon_ward'], groups=['post'], fish_rates=RF_POST,
    param_combos=COMBOS, years=60, spinup=15, n_harmonics=3, forcings=FORC, routed=True,
    save_path='seasonal_grid2_post_2026-06-29.pkl', processes=None, maxtasksperchild=1)
print('post done:', len(res.records))

360 constructs x 3 r_F = 1080 post runs
[seasonal scan] 1080 runs: ['maranon_ward'] x ['post'] x fish=[0.1, 0.15, 0.2] x params['GGE', 'KsZ', 'fish_fD', 'fish_fX', 'k_remin', 'mP', 'm_Z', 'sigma_log', 'w_sink', 'zl_fD', 'zl_fX', 'zq_fD', 'zq_fX'] | 60 yr each (spin-up 15)
solver: {'method': 'RK45', 'atol': 1e-09, 'rtol': 1e-06, 'max_step': 1.0, 'instability_neg_threshold': -0.001} | fourier n_harm=3 | processes=cpu-1
  MODEL   : model_baseline_seasonal_routed   (ROUTED closure+fish)
  growth  : maranon_ward  mu_max[0.13-0.83]  Ks[0.04-10.45]
  grazing : KsZ 0.15  sigma_log 0.15  Q10 2.48
  scalars : GGE 0.25  growth-Q10 1.62  T_ref 20.0
  rates   : mP 0.0015  m_Z(quad) 0.05  m_Zlin 0.05  r_F 0.1
  detritus: k_remin 0.04  w_sink 5.0  ->  L = 125 m
  loss-fate routing (N / D / export):
     phyto mort : N 0.10 / D 0.90 / export 0.00
     graze unas : N 0.25 / D 0.75 / export 0.00
     zoo linear : N 0.10 / D 0.90 / export 0.00
     zoo quad   : N 0.10 / D 0.90 / export 0.00
     fish    

In [5]:
import numpy as np, pandas as pd, seasonal_scan_harness as ssh
from collections import defaultdict
RP=ssh.load_results('seasonal_grid2_pre_2026-06-29.pkl'); RQ=ssh.load_results('seasonal_grid2_post_2026-06-29.pkl')
OBS={'pre+recovery':RP.obs['pre+recovery'],'post':RQ.obs['post']}; obs_m=ssh.build_obs_monthly(['pre+recovery','post']); _MO=np.arange(1,13)
log_oc={g:np.log10(obs_m[g].dropna(subset=['mcs']).groupby('mo')['mcs'].median().reindex(_MO).to_numpy()) for g in OBS}
oP,oQ=OBS['pre+recovery']['med'],OBS['post']['med']
CK=['KsZ','GGE','sigma_log','m

    c=np.asarray(r.get('clim_mcs',[np.nan]*12),float); return float(np.sqrt(np.nanmean((np.log10(c)-log_oc[g])**2)))
def rs(a,b): return (a-b)/a if abs(a)>1e-9 else np.nan
Qby=defaultdict(list)
for q in RQ.records: Qby[ck(q)].append(q)
rows=[]
for p in RP.records:
    if p.get('has_nan'): continue
    for q in Qby.get(ck(p),[]):
        if q.get('has_nan') or q['fish']>p['fish']+1e-9: continue
        comp=max(comp_e(p,oP),comp_e(q,oQ)); bl=max(brmse(p,'pre+recovery'),brmse(q,'post'))
        inv=q['Z200_med']-p['Z200_med']; Zrat=q['Z200_med']/max(p['Z200_med'],1e-9)
        shP=abs(rs(p['sumP_med'],q['sumP_med'])-rs(oP['sumP'],oQ['sumP'])); shN=abs(rs(p['N_med'],q['N_med'])-rs(oP['N'],oQ['N']))
        sqg=p.get('squiggle_sumP',np.nan)
        row={k:round(p[k],3) for k in CK}
        row.update(rFpre=round(p['fish'],2),rFpost=round(q['fish'],2), score=round(comp+bl,3),
            comp=round(comp,3), bloom=round(bl,3), squig=round(sqg,3), inv=round(inv,4), Zrat=round(Zrat,2),
            shP=round(shP,3), shN=round(shN,3), pze=round(p.get('pze',np.nan),2), cvPre=round(p['cv_sumP'],2),
            sq_ok=bool(sqg<=SQ_THR), Z_ok=bool(inv>=ZMIN_INV and p['Z200_med']>=ZPRE_FLOOR and Zrat<=ZRAT_MAX),
            shP_ok=bool(shP<=SHP_TOL), shN_ok=bool(shN<=SHN_TOL))
        rows.append(row)
X=pd.DataFrame(rows); X['flags']=X.sq_ok&X.Z_ok&X.shP_ok&X.shN_ok
cols=CK+['rFpre','rFpost','score','comp','bloom','squig','inv','Zrat','shP','shN','pze','cvPre','sq_ok','Z_ok','shP_ok','shN_ok']
print(f"OBS relshift sumP {rs(oP['sumP'],oQ['sumP']):.2f} N {rs(oP['N'],oQ['N']):.2f} | {len(X)} stable pairs | "
      f"flags sq<={SQ_THR} Z(inv>={ZMIN_INV},Zpre>={ZPRE_FLOOR},rat<={ZRAT_MAX}) shP<={SHP_TOL} shN<={SHN_TOL}")
print(f"\n=== TOP 25 — ALL pairs by score (comp+bloom), flags shown ===")
print(X.sort_values('score')[cols].head(25).to_string(index=False))
print(f"\n=== TOP 25 — FLAG-PASSING ({int(X.flags.sum())}/{len(X)}) by score ===")
print((X[X.flags].sort_values('score')[cols].head(25).to_string(index=False)) if X.flags.any() else "  none pass all flags — relax a threshold (likely shN) and re-read")

OBS relshift sumP 0.17 N 0.25 | 6471 stable pairs | flags sq<=0.3 Z(inv>=0.002,Zpre>=0.008,rat<=2.0) shP<=0.15 shN<=0.25

=== TOP 25 — ALL pairs by score (comp+bloom), flags shown ===
 KsZ  GGE  sigma_log  m_Z  w_sink  k_remin  rFpre  rFpost  score  comp  bloom  squig     inv  Zrat   shP   shN  pze  cvPre  sq_ok  Z_ok  shP_ok  shN_ok
0.15 0.33       0.22 0.10     5.0    0.067   0.65    0.20  0.528 0.134  0.394  0.045 -0.0130  0.81 0.178 0.442 0.48   0.17   True False   False   False
0.15 0.25       0.20 0.10     5.0    0.067   0.65    0.10  0.538 0.078  0.460  0.516  0.0038  1.22 0.156 0.137 0.49   1.38  False  True   False    True
0.15 0.25       0.20 0.10     5.0    0.067   0.55    0.10  0.538 0.078  0.460  0.494  0.0042  1.24 0.149 0.119 0.49   1.35  False  True    True    True
0.15 0.31       0.22 0.10     5.0    0.067   0.65    0.15  0.538 0.159  0.379  0.044 -0.0120  0.82 0.192 0.424 0.48   0.17   True False   False   False
0.15 0.31       0.22 0.10     5.0    0.067   0.60    0.1

AttributeError: 'Flags' object has no attribute 'sum'